# 3D Loss Landscape

A 3D loss landscape shows loss as height over two directions in parameter space. The real model has many more dimensions, so this is still a slice, not the full landscape.

In [ ]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()
device

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)

train_loader = DataLoader(Subset(train_data, range(6000)), batch_size=128, shuffle=True)
landscape_loader = DataLoader(Subset(test_data, range(256)), batch_size=256)
landscape_images, landscape_labels = next(iter(landscape_loader))
landscape_images, landscape_labels = landscape_images.to(device), landscape_labels.to(device)

In [ ]:
def build_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10),
    ).to(device)

model = build_model()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(3):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

trained_state = copy.deepcopy(model.state_dict())

In [ ]:
def random_direction_like(state):
    direction = {}
    for name, value in state.items():
        d = torch.randn_like(value)
        d = d / (d.norm() + 1e-8) * (value.norm() + 1e-8)
        direction[name] = d
    return direction

direction_a = random_direction_like(trained_state)
direction_b = random_direction_like(trained_state)

def state_at(alpha, beta):
    return {
        name: trained_state[name] + alpha * direction_a[name] + beta * direction_b[name]
        for name in trained_state
    }

def batch_loss_at(alpha, beta):
    model.load_state_dict(state_at(alpha, beta))
    model.eval()
    with torch.no_grad():
        return loss_fn(model(landscape_images), landscape_labels).item()

In [ ]:
grid = torch.linspace(-0.7, 0.7, 31)
losses = torch.zeros(len(grid), len(grid))

for i, alpha in enumerate(grid):
    for j, beta in enumerate(grid):
        losses[i, j] = batch_loss_at(alpha.item(), beta.item())

model.load_state_dict(trained_state)

X, Y = torch.meshgrid(grid, grid, indexing="ij")

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surface = ax.plot_surface(X.numpy(), Y.numpy(), losses.numpy(), cmap="viridis", linewidth=0, antialiased=True, alpha=0.92)
ax.scatter([0], [0], [batch_loss_at(0, 0)], color="red", s=45, label="trained weights")
ax.set_xlabel("direction A")
ax.set_ylabel("direction B")
ax.set_zlabel("cross entropy loss")
ax.set_title("3D loss landscape slice")
ax.view_init(elev=30, azim=135)
fig.colorbar(surface, shrink=0.65, label="loss")
ax.legend()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(7, 6))
contours = plt.contourf(X, Y, losses, levels=30, cmap="viridis")
plt.colorbar(contours, label="cross entropy loss")
plt.scatter([0], [0], color="red", label="trained weights")
plt.xlabel("direction A")
plt.ylabel("direction B")
plt.title("Same loss landscape as a contour map")
plt.legend()
plt.tight_layout()

## Exercise

Change `ax.view_init`, the grid range, or the number of training epochs. How does the surface change?